# PathSnap: 01. Exploratory Data Analysis (EDA)
### GPS Trajectory Map-Matching & Road Network Characterization

This notebook performs exploratory data analysis on the PathSnap dataset:
1. **Dataset Inventory & Integrity Verification**: File schema, coordinate validation, missing values.
2. **Kinematic & Temporal Analysis**: Nominal 1 Hz sampling verification ($\Delta t$), instantaneous speed distribution.
3. **Road Network Graph Analysis**: Nodes, directional arcs, degree distribution.
4. **Geospatial Visualization**: Interactive Leaflet/Folium map rendering raw GPS observations.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to sys.path to access src module
sys.path.append(str(Path(".").resolve().parent))

from src.data.loader import TripLoader
from src.data.preprocessing import haversine_distance, compute_bearing
from src.visualization.maps import create_folium_map, save_folium_map

loader = TripLoader("../data")
train_trips = loader.list_trips("train")
test_trips = loader.list_trips("student_test")

print(f"Loaded {len(train_trips)} training trips and {len(test_trips)} student-test trips.")


## 1. Dataset Inventory & Schema Verification
Each trip contains 4 coordinate and topological files:
- `.track`: [Longitude, Latitude, Timestamp in seconds]
- `.nodes`: [Longitude, Latitude] for road intersections
- `.arcs`: [from_node_id, to_node_id] directed road segments
- `.route`: Chronological sequence of ground-truth traversed arc IDs


In [ ]:
summary_records = []
for split, trips in [("train", train_trips[:15]), ("student_test", test_trips)]:
    for tid in trips:
        trip = loader.load_trip(tid, split=split)
        val = loader.validate_trip(trip)
        summary_records.append({"split": split, "trip_id": tid, **val})

df_summary = pd.DataFrame(summary_records)
print(f"Validated {len(df_summary)} trips across splits.")
df_summary[["split", "trip_id", "num_observations", "num_nodes", "num_arcs", "nominal_1hz_pct"]].head(10)


## 2. Temporal & Kinematic Distributions
We analyze sampling intervals ($\Delta t$) and instantaneous speeds ($v = \Delta d / \Delta t$) using spherical Haversine distances.


In [ ]:
sample_trip = loader.load_trip(test_trips[0], split="student_test")
track = sample_trip["track"]
dts = np.diff(track[:, 2])
speeds = []
for i in range(len(track) - 1):
    # track column 0 is lon, column 1 is lat
    dist = haversine_distance(track[i, 1], track[i, 0], track[i+1, 1], track[i+1, 0])
    speeds.append(dist / max(1e-3, dts[i]))

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.hist(dts, bins=25, color="#3B82F6", edgecolor="black")
plt.title("GPS Sampling Intervals (Δt)")
plt.xlabel("Seconds")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
plt.hist(speeds, bins=25, color="#10B981", edgecolor="black")
plt.title("Instantaneous Speeds (m/s)")
plt.xlabel("Speed (m/s)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 3. Interactive Geospatial Visualization
Renders the GPS trajectory and road network using Folium / Leaflet with OpenStreetMap tiles (requires no API keys).


In [ ]:
fmap = create_folium_map(track=sample_trip["track"], trip_id=test_trips[0])
save_folium_map(fmap, "../results/plots/eda_sample_map.html")
print("Interactive map successfully generated and saved to results/plots/eda_sample_map.html")
